# ABS Wages

Through-the-year growth in three measures of Australian pay, on one axis.

The three are not the same object, and the gaps between the lines are mostly
that rather than disagreement about wages:

- **WPI** (Wage Price Index, ABS 6345.0) prices a *fixed basket of jobs*, total
  hourly rates of pay excluding bonuses. It is built to strip out composition:
  if the workforce shifts towards better-paid jobs, the WPI does not move. It is
  the cleanest read on what an employer pays for the same work, and the narrowest.
- **AWOTE** (average weekly ordinary-time earnings, ABS 6302.0) is an actual
  average of a *subgroup* - full-time adults, ordinary time only. It moves with
  composition, and it says nothing about part-time or overtime pay. Biannual.
- **COE per hour** (compensation of employees per hour, ABS 5206.0 national
  accounts) is the whole employee wage bill - wages and salaries *plus* super
  and other employer contributions - divided by hours worked. It is a labour
  *cost* concept, it includes non-wage costs, and its denominator counts the
  self-employed while its numerator does not, so it is a ratio rather than
  anyone's pay. It is also the noisiest of the three.

Read together: WPI is the signal on wage *rates*, COE per hour on labour
*costs*, and AWOTE on what a full-time worker actually banks.

## Python set-up

In [1]:
# analytic imports
import mgplot as mg
import pandas as pd
import readabs as ra
from pandas import DataFrame, Series
from readabs import metacol as mc

# local imports
from abs_prices import get_wage_index

# pandas display settings
pd.options.display.max_rows = 999999
pd.options.display.max_columns = 999
pd.options.display.max_colwidth = 100

In [2]:
# Constants
SHOW = False
FILE_TYPE = "png"

source = "ABS 6345.0, 6302.0, 5206.0"
PAIR_SOURCE = "ABS 6345.0, 6302.0"  # no national accounts once COE is dropped

# The three measures do not share a series type: the WPI and the national
# accounts series are published Seasonally Adjusted, AWOTE only Original.
LFOOTER = "Australia. WPI and COE per hour seasonally adjusted; AWOTE original. "

# The legend labels are acronyms; spell them out above the chart.
WPI_GLOSS = "WPI = Wage Price Index"
AWOTE_GLOSS = "AWOTE = Average Weekly Ordinary Time Earnings"
COE_GLOSS = "COE = Compensation of Employees"
HEADER = f"{WPI_GLOSS}; {AWOTE_GLOSS}; {COE_GLOSS}"
PAIR_HEADER = f"{WPI_GLOSS}; {AWOTE_GLOSS}"

# The two directly comparable measures: both are employee pay, and neither has
# COE per hour's ratio construction or its compositional volatility.
PAIR = ["WPI", "AWOTE"]

# Full history, and the last 45 quarters (a bit over a decade).
plot_times = 0, -45

# Quarters in a year - the lag used for through-the-year growth.
QUARTERS_PER_YEAR = 4

# Compensation of employees per hour lives in the national accounts analytical
# series table, not the key aggregates.
COE_TABLE = "5206024_Selected_Analytical_Series"
COE_SELECTOR = {
    COE_TABLE: mc.table,
    "Compensation of employees per hour: Current prices ;": mc.did,
    "Seasonally Adjusted": mc.stype,
}

# Output chart directory (fresh for this notebook).
mg.set_chart_dir("./CHARTS/Wages/")
mg.clear_chart_dir()

## Through-the-year growth on a gappy index

AWOTE is biannual but carried on a quarterly `PeriodIndex`, occupying only Q2
and Q4. A positional `pct_change(4)` would therefore reach back *two* years for
AWOTE while reaching back one for the other two, and say nothing about it.

So growth is taken by shifting the index *labels* forward a year and dividing on
alignment. That is frequency-honest whatever the spacing: each observation is
compared with the one exactly four quarters earlier, and where no such
observation exists the result is NaN rather than a quietly wrong number.

In [3]:
def through_the_year(series: Series, lag: int = QUARTERS_PER_YEAR) -> Series:
    """Return annual growth in per cent, matching on the period index.

    Unlike a positional pct_change(), this compares each observation with the
    one exactly `lag` quarters earlier, so it is correct for a series that does
    not occupy every quarter (AWOTE occupies only Q2 and Q4).

    Args:
        series: a level or index, on a quarterly PeriodIndex.
        lag: the comparison lag, in quarters.

    Returns:
        The through-the-year growth rate, in per cent, with NaN dropped.

    """
    index = series.index
    if not isinstance(index, pd.PeriodIndex):
        raise TypeError(f"Expected a PeriodIndex, got {type(index).__name__}")

    year_earlier = series.copy()
    year_earlier.index = index + lag
    return ((series / year_earlier - 1) * 100).dropna()

## Fetch the three measures

In [4]:
def get_coe_per_hour() -> Series:
    """Return seasonally adjusted compensation of employees per hour ($/hour).

    Returns:
        The series, from the 5206.0 selected analytical series table.

    """
    data, meta = ra.read_abs_cat("5206.0", single_excel_only=COE_TABLE)
    _, series_id, _ = ra.find_abs_id(meta, COE_SELECTOR)
    series = data[COE_TABLE][series_id].dropna()
    if series.empty:
        raise ValueError(f"No data for compensation of employees per hour ({series_id})")
    return series


def get_wage_growth() -> DataFrame:
    """Assemble through-the-year growth for the three wage measures.

    Returns:
        A DataFrame on a quarterly PeriodIndex, one column per measure. AWOTE is
        biannual, so its column is NaN in the intervening quarters.

    """
    wpi, _units, _stype = get_wage_index("WPI")
    awote, _units, _stype = get_wage_index("AWOTE")
    levels = {"WPI": wpi, "AWOTE": awote, "COE per hour": get_coe_per_hour()}
    return DataFrame({name: through_the_year(series) for name, series in levels.items()})

In [5]:
growth = get_wage_growth()
growth.tail(8).round(2)

,WPI,AWOTE,COE per hour
Series ID,,,
2024Q3,3.59,NaN,3.81
2024Q4,3.22,4.61,3.38
2025Q1,3.46,NaN,3.87
2025Q2,3.43,4.50,4.67
2025Q3,3.34,NaN,5.83
2025Q4,3.44,3.81,4.42
2026Q1,3.22,NaN,3.40
2026Q2,3.19,3.67,NaN


## Plot

In [6]:
def plot_wage_growth(
    growth: DataFrame,
    title: str,
    lfooter: str = LFOOTER,
    lheader: str = HEADER,
    rfooter: str = source,
) -> None:
    """Plot through-the-year growth in the wage measures.

    Args:
        growth: a growth DataFrame from get_wage_growth(), or a subset of one.
        title: the chart title.
        lfooter: the left footer.
        lheader: the left header, glossing the acronyms in the legend.
        rfooter: the right footer, attributing the sources actually plotted.

    """
    mg.multi_start(
        growth,
        function=mg.line_plot_finalise,
        starts=plot_times,
        title=title,
        ylabel="Per cent per year",
        y0=True,
        legend={"loc": "best", "fontsize": "small"},
        lheader=lheader,
        rfooter=rfooter,
        lfooter=lfooter,
        show=SHOW,
        file_type=FILE_TYPE,
    )

In [7]:
plot_wage_growth(growth, title="Wages: Annual Growth in Three Measures")

### Just the two comparable measures

COE per hour is a ratio of two volatile national accounts aggregates whose
numerator and denominator cover different populations, and its largest swings
are compositional rather than pay. Dropping it leaves the two series that are
both straightforwardly employee pay - and the chart is legible without any
filtering.

In [8]:
plot_wage_growth(
    growth[PAIR],
    title="Wages: Annual Growth in the WPI and AWOTE",
    lfooter="Australia. WPI seasonally adjusted; AWOTE original. ",
    lheader=PAIR_HEADER,
    rfooter=PAIR_SOURCE,
)

## Watermark

In [9]:
%load_ext watermark
%watermark -u -t -d --iversions --watermark
print("Finished")

Last updated: 2026-08-20 10:11:07

mgplot : 0.2.31
pandas : 3.0.5
readabs: 0.2.5

Watermark: 2.6.0

Finished
